In [ ]:
#2.2 Working with Text Data
!pip install tiktoken

In [ ]:
import os
import urllib.request

#fetching the dataset

if not os.path.exists('the-verdict.txt'):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"
    file_path ='the-verdict.txt'
    urllib.request.urlretrieve(url, file_path)


In [ ]:
with open ( "the-verdict.txt","r",encoding="utf-8") as f:
  raw_text = f.read()
raw_text[:1000]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [ ]:
len(raw_text)

20479

In [ ]:
#converting to tokens:
import re
pattern = r"""([,:;?!'"()]|--|\s)"""


#testing
text ="Hello, world, Is this a test?"
tokens = re.split(pattern,text)
print(tokens)

['Hello', ',', '', ' ', 'world', ',', '', ' ', 'Is', ' ', 'this', ' ', 'a', ' ', 'test', '?', '']


In [ ]:
#cleaning tokens manually:

piece = [token for token in tokens if token.split()]
text_tokens = re.split(pattern,raw_text)
preprocessed = [token for token in text_tokens if token.strip()]

In [ ]:

#building vocab:
unique_tokens = sorted(set(preprocessed))

vocab = {
    token: token_id
    for token_id, token in enumerate(unique_tokens)
}
vocab_size = len(vocab)
print(list(vocab.items())[:10])

[('!', 0), ('"', 1), ("'", 2), ('(', 3), (')', 4), (',', 5), ('--', 6), ('.', 7), (':', 8), (';', 9)]


In [ ]:
class SimpleTokenizerV1:
  def __init__(self,vocab):
     self.str_to_int = vocab
     self.int_to_str = {
         token_id: token
         for token, token_id in vocab.items()
     }

  def encode(self, text):
    pieces = re.split(pattern, text)
    tokens = [token for token in pieces if token.strip()]
    return [self.str_to_int[token] for token in tokens]


  def decode(self, ids):
    # looking up each ID's token.
    tokens = [self.int_to_str[token_id] for token_id in ids]

    text = " ".join(tokens)
    return text
manual_tokenizer = SimpleTokenizerV1(vocab)




sample = " ".join(preprocessed[:20])

ids = manual_tokenizer.encode(sample)

print("IDs:", ids)
print("Decoded:", manual_tokenizer.decode(ids))

IDs: [54, 45, 171, 1085, 58, 38, 883, 133, 284, 523, 6, 1084, 133, 537, 470, 427, 6, 979, 629, 1164]
Decoded: I HAD always thought Jack Gisburn rather a cheap genius -- though a good fellow enough -- so it was


In [ ]:
# adding special tokens. for eg- hello is not in our vocabulary
special_tokens = [

    "<|endoftext|>",
    "<|startoftext|>",
    "<|endofchunk|>"]

vocab_v2 = vocab.copy()

for token in special_tokens:
  if token not in vocab_v2:
    vocab_v2[token] = len(vocab_v2)

In [ ]:
class SimpleTokenizerV2(SimpleTokenizerV1):


  def encode(self, text):
    pieces = re.split(pattern, text)
    tokens = [token for token in pieces if token.strip()]
    tokens = [
        token if token in self.str_to_int else "<|unk|>"
        for token in tokens
    ]
    return [self.str_to_int[token] for token in tokens]



tokenizer = SimpleTokenizerV2(vocab_v2)


In [ ]:
#using tiktoken for Byte Pair Encoding
import tiktoken
#what it does is it has a encoding algo of itself, and also maintains itself a vocabulary, pretrained.

tokenizer = tiktoken.get_encoding("gpt2")

#testing
text = "Hello, do you like tea?"
token_ids = tokenizer.encode(text)

print("IDs:", token_ids)
print("Decoded:", tokenizer.decode(token_ids))

IDs: [15496, 11, 466, 345, 588, 8887, 30]
Decoded: Hello, do you like tea?


ALL UNDERSTOOD UPTO HERE, next is understanding inputs and outputs for next token prediction

In [ ]:
example_ids = [10, 20, 30, 40, 50, 60]
max_length = 4
stride = 2

inputs = example_ids[:max_length]
targets = example_ids[1:max_length + 1]


print("Inputs: ", inputs)   # [10, 20, 30, 40]
print("Targets:    ", targets)


for position in range(max_length):
    context = inputs[:position+1]
    next_token = targets[position]

Inputs:  [10, 20, 30, 40]
Targets:     [20, 30, 40, 50]


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):

    def __init__(self, text, tokenizer, max_length, stride):
      self.input_ids = []
      self.target_ids = []

      token_ids = tokenizer.encode(
          text,
          allowed_special={'<|endoftext|>'}
      )

      for i in range(0,len(token_ids)-max_length, stride): #we subtract to have enough token for the last step

          input_chunk = token_ids[i:i+max_length]
          target_chunk = token_ids[i+1:i+max_length+1]
        # Convert lists to integer tensors for PyTorch.
          self.input_ids.append(
              torch.tensor(input_chunk, dtype=torch.long)
          )
          self.target_ids.append(
              torch.tensor(target_chunk, dtype=torch.long)
          )
    def __len__(self):
      return len(self.input_ids)

    def __getitem__(self, idx):
      return self.input_ids[idx], self.target_ids[idx]

#creating dataset from the above class
dataset = GPTDatasetV1(
    raw_text,
    tokenizer,
    max_length=4,
    stride=1
)





In [ ]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle = True,
    drop_last = True,
    num_workers = 0
):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(
        txt,
        tokenizer,
        max_length=max_length,
        stride=stride)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

#making a dataloader


dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=True,
    drop_last=True,
    num_workers=0
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs:\n", inputs)
print("Targets:\n", targets)

print("Input shape:", inputs.shape)    # [8, 4]
print("Target shape:", targets.shape)  # [8, 4]

Inputs:
 tensor([[  383,  5019, 19001,   286],
        [  286,   262, 34686, 41976],
        [19133,   553,   373,   530],
        [  625,   422, 22489, 40089],
        [  616,  2951,   625,     0],
        [  373,  2636,    30,   921],
        [  326,   612,   550,   587],
        [ 1061,   510,   428, 18437]])
Targets:
 tensor([[ 5019, 19001,   286,   262],
        [  262, 34686, 41976, 37733],
        [  553,   373,   530,   286],
        [  422, 22489, 40089,    26],
        [ 2951,   625,     0, 19672],
        [ 2636,    30,   921,  1701],
        [  612,   550,   587,   645],
        [  510,   428, 18437,   618]])
Input shape: torch.Size([8, 4])
Target shape: torch.Size([8, 4])


In [ ]:
vocab_size = tokenizer.n_vocab
embedding_dim = 256


token_embedding_layer = torch.nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)


token_embeddings = token_embedding_layer(inputs)

print("Token embeddings shape:", token_embeddings.shape)

#now making
context_length = inputs.shape[1]


pos_embedding_layer = torch.nn.Embedding(
    context_length,
    embedding_dim
)

pos_ids = torch.arange(context_length, dtype=torch.long)
pos_embeddings = pos_embedding_layer(pos_ids).unsqueeze(0)

print("Positional embeddings shape:", pos_embeddings.shape)

Token embeddings shape: torch.Size([8, 4, 256])
Positional embeddings shape: torch.Size([1, 4, 256])


In [ ]:
#combining both -- RoPE is the standard, this is the gpt 2 one, where it populates the input embeddings instead of the Query Vectors

In [ ]:
input_embeddings = token_embeddings + pos_embeddings

input_embeddings.shape

torch.Size([8, 4, 256])